# Spectral Aware Quantization

1. Loads Chronos-2 and builds an eager-attention `MaseGraph`
2. Produces a layer/node inventory for the FX graph
3. Runs a baseline integer PTQ flow on the full Chronos-2 graph
4. Profiles activation spectra and builds a spectral-aware PTQ config
5. Benchmarks FP32 vs baseline PTQ vs spectral-aware PTQ with FEV

In [ ]:
from huggingface_hub import login
login()  

In [ ]:
import io
import math
import os
import time
from itertools import islice
from pathlib import Path

import lightning as pl
import pandas as pd
import requests
import torch
import tqdm
import shutil
import gc


os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))

import fev
from chronos import Chronos2Pipeline

from torch.utils.data import DataLoader

from chop.dataset import get_dataset_info
from chop.ir.graph import MaseGraph
from chop.dataset.timeseries import get_timeseries_dataset
from chop.models import get_model, get_model_info
from chop.models.chronos2 import (
    CHRONOS2_FX_INPUT_NAMES,
    attach_chronos2_graph_module_metadata,
    build_chronos2_dummy_input,
    build_chronos2_mase_graph,
    build_spectral_quant_config,
    chronos2_node_inventory,
    force_eager_attention_for_fx,
    make_integer_quant_config,
)
from chop.passes.graph import PASSES
from chop.passes.graph.analysis.quantization import spectral_calibrate_quantization
from chop.tools.get_input import get_dummy_input, get_hf_input_names
from chop.tools.plt_wrapper import get_model_wrapper


In [ ]:
MODEL_NAME = "chronos-2"
CALIBRATION_DATASET = "chronos_m4_daily"
TASK = "forecasting"
DEVICE = os.environ.get("MASE_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
CALIBRATION_BATCHES = 8
TIME_SERIES_BATCH_SIZE = 4
TIME_SERIES_NUM_WORKERS = 0
TIME_SERIES_CONTEXT_LENGTH = 512
TIME_SERIES_PREDICTION_LENGTH = None  # use dataset default
TIME_SERIES_STRIDE = 1
OUTPUT_DIR = Path("artifacts/chronos2_spectral_workflow")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_FEV_REPO = "autogluon/fev"
OFFICIAL_FEV_BENCHMARK_REF = "main"
OFFICIAL_FEV_RESULTS_PATH = "benchmarks/fev_bench/results"
OFFICIAL_FEV_BENCHMARK_PATH_CANDIDATES = [
    "benchmarks/fev_bench/benchmark.yaml",
    "benchmarks/fev_bench/benchmark.yml",
    "benchmarks/fev_bench/tasks.yaml",
    "benchmarks/fev_bench/tasks.yml",
]
OFFICIAL_FEV_BASELINE_MODEL = "Seasonal Naive"
OFFICIAL_FEV_LEAKAGE_IMPUTATION_MODEL = "Chronos-Bolt"
OFFICIAL_FEV_LEADERBOARD_METRICS = ["SQL", "MASE", "WQL", "WAPE"]
OFFICIAL_FEV_NORMALIZE_TIME_PER_N_FORECASTS = 100
FEV_BENCH_TASK_LIMIT = None
OFFICIAL_FEV_ALLOW_NETWORK = False
OFFICIAL_FEV_USE_LOCAL_HF_CACHE = True
OFFICIAL_FEV_SKIP_UNCACHED_TASKS = True
OFFICIAL_FEV_BENCHMARK_CACHE_PATH = OUTPUT_DIR / f"official_fev_benchmark_{OFFICIAL_FEV_BENCHMARK_REF}.yaml"
HF_DATASETS_CACHE_ROOT = Path(os.environ.get("HF_DATASETS_CACHE", Path.home() / ".cache" / "huggingface" / "datasets"))

BASE_INTEGER_CONFIG = make_integer_quant_config(
    data_in_width=8,
    data_in_frac_width=4,
    weight_width=8,
    weight_frac_width=4,
    bias_width=8,
    bias_frac_width=4,
)

def clone_integer_config(**overrides):
    config = dict(BASE_INTEGER_CONFIG)
    config.update(overrides)
    return config

def make_baseline_quant_pass_args():
    return {
        "by": "type",
        "default": {"config": {"name": None}},
        "linear": {"config": clone_integer_config()},
        "matmul": {
            "config": {
                "name": "integer",
                "data_in_width": 8,
                "data_in_frac_width": 4,
                "weight_width": 8,
                "weight_frac_width": 4,
            }
        },
    }

def make_spectral_quant_pass_args(mase_graph, frac_width_by_module):
    pass_args = build_spectral_quant_config(
        mase_graph,
        frac_width_by_module,
        default_config=BASE_INTEGER_CONFIG,
    )
    # Use exact node-name matching to avoid the regex quantization path,
    # which deep-copies Chronos2 metadata and fails on non-leaf tensors.
    # Also disable the default quantizer so only explicitly calibrated nodes
    # are quantized in the spectral run.
    pass_args["by"] = "name"
    pass_args["default"] = {"config": {"name": None}}
    return pass_args


In [ ]:
def load_fresh_chronos2(model_name=MODEL_NAME, device=DEVICE):
    model = get_model(model_name, pretrained=True)
    force_eager_attention_for_fx(model)
    model.eval()
    if device:
        if str(device).startswith("cuda") and not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but torch.cuda.is_available() is False")
        model = model.to(device)
    return model

model = load_fresh_chronos2()
print("Loaded model:", MODEL_NAME)
print(model.config.chronos_config)


In [ ]:
inference_dummy_input = build_chronos2_dummy_input(model, batch_size=1, device=DEVICE)
mg = build_chronos2_mase_graph(
    model=model,
    dummy_in=inference_dummy_input,
    hf_input_names=CHRONOS2_FX_INPUT_NAMES,
)
print(f"Graph nodes before quantization: {sum(1 for _ in mg.nodes)}")

In [ ]:
inventory = chronos2_node_inventory(mg)
inventory_frames = {
    name: pd.DataFrame(rows) if rows else pd.DataFrame(columns=["name", "op", "target", "mase_op", "module_type"])
    for name, rows in inventory.items()
}

for group_name, frame in inventory_frames.items():
    print(f"[{group_name}] count = {len(frame)}")
    display(frame.head(20))

In [ ]:
def select_spectral_layers(module: torch.nn.Module) -> list[str]:
    tokens = (
        "input_patch_embedding",
        "output_patch_embedding",
        "self_attention.q",
        "self_attention.k",
        "self_attention.v",
        "self_attention.o",
    )
    selected = []
    for name, submodule in module.named_modules():
        if isinstance(submodule, torch.nn.Linear) and any(token in name for token in tokens):
            selected.append(name)
    return selected

spectral_target_layers = select_spectral_layers(model)
print("Selected spectral calibration layers:")
for layer_name in spectral_target_layers:
    print(" -", layer_name)

In [ ]:
baseline_model = load_fresh_chronos2()
baseline_dummy_input = build_chronos2_dummy_input(baseline_model, batch_size=1, device=DEVICE)
baseline_mg = build_chronos2_mase_graph(
    model=baseline_model,
    dummy_in=baseline_dummy_input,
    hf_input_names=CHRONOS2_FX_INPUT_NAMES,
)
baseline_mg, _ = PASSES["quantize"](baseline_mg, pass_args=make_baseline_quant_pass_args())
print("Baseline graph PTQ complete")


In [ ]:
train_ds = get_timeseries_dataset(
    CALIBRATION_DATASET,
    split="train",
    context_length=TIME_SERIES_CONTEXT_LENGTH,
    prediction_length=TIME_SERIES_PREDICTION_LENGTH,
    stride=TIME_SERIES_STRIDE,
    auto_setup=False,
)
train_ds.prepare_data()
train_ds.setup()

train_loader = DataLoader(
    train_ds,
    batch_size=TIME_SERIES_BATCH_SIZE,
    shuffle=True,
    num_workers=TIME_SERIES_NUM_WORKERS,
)
output_patch_size = int(model.chronos_config.output_patch_size)

def make_inference_batch(batch: dict) -> dict:
    past_values = batch["past_values"].to(DEVICE)
    future_values = batch["future_values"]
    num_output_patches = max(1, math.ceil(future_values.shape[-1] / output_patch_size))
    future_length = num_output_patches * output_patch_size
    return {
        "context": past_values,
        "group_ids": torch.zeros((past_values.shape[0],), dtype=torch.long, device=DEVICE),
        "future_covariates": torch.zeros((past_values.shape[0], future_length), dtype=torch.float32, device=DEVICE),
        "num_output_patches": num_output_patches,
    }

calibration_source_batches = list(islice(iter(train_loader), CALIBRATION_BATCHES))
calibration_batches = [make_inference_batch(batch) for batch in calibration_source_batches]

print(train_ds)
print(f"Prepared {len(calibration_batches)} calibration batches from {CALIBRATION_DATASET}")
print(f"Context length: {train_ds.context_length}, prediction length: {train_ds.prediction_length}, stride: {train_ds.stride}")


In [ ]:
_, spectral_meta = PASSES["profile_spectral_statistics"](
    mg,
    pass_args={
        "calibration_batches": calibration_batches,
        "max_batches": CALIBRATION_BATCHES,
        "target_layers": spectral_target_layers,
        "num_bands": 8,
        "max_samples_per_layer": 4,
    },
)

spectral_stats = spectral_meta["spectral_stats"]
frac_width_by_module = spectral_calibrate_quantization(
    mg,
    spectral_stats,
    target_layers=spectral_target_layers,
    width=8,
    bias_width=8,
    default_frac_width=4,
    frac_width_candidates=list(range(8)),
    num_bands=8,
)

spectral_df = pd.DataFrame.from_dict(frac_width_by_module, orient="index")
display(spectral_df.sort_index())

In [ ]:
spectral_target_node_count = len(frac_width_by_module)
print(f"Prepared spectral quant config template for {spectral_target_node_count} target modules")
print("Spectral quantization will use exact node-name matching (by='name')")


In [ ]:
spectral_model = load_fresh_chronos2()
spectral_dummy_input = build_chronos2_dummy_input(spectral_model, batch_size=1, device=DEVICE)
spectral_mg = build_chronos2_mase_graph(
    model=spectral_model,
    dummy_in=spectral_dummy_input,
    hf_input_names=CHRONOS2_FX_INPUT_NAMES,
)
spectral_mg, _ = PASSES["quantize"](spectral_mg, pass_args=make_spectral_quant_pass_args(mg, frac_width_by_module))
print("Spectral-aware PTQ complete")


In [ ]:
for label, graph in {
    "baseline": baseline_mg,
    "spectral": spectral_mg,
}.items():
    base = OUTPUT_DIR / f"chronos2_{label}"
    graph.export(str(base))
    print(f"Exported {label} graph to {base}.pt / {base}.mz")

In [ ]:
def build_pipeline_from_graph(graph: MaseGraph, reference_model) -> Chronos2Pipeline:
    attach_chronos2_graph_module_metadata(graph.model, reference_model, wrap_forward=True)
    return Chronos2Pipeline(model=graph.model)


def http_get_with_retries(url: str, timeout: int = 20, attempts: int = 3, **kwargs) -> requests.Response:
    last_exc = None
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(url, timeout=timeout, **kwargs)
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_exc = exc
            if attempt == attempts:
                raise
            time.sleep(min(2 ** (attempt - 1), 5))
    raise RuntimeError(f"Unexpected request retry flow for {url}: {last_exc}")


def resolve_cached_hf_dataset_file(repo_id: str, config: str) -> str | None:
    config_root = HF_DATASETS_CACHE_ROOT / repo_id.replace("/", "___") / config
    if not config_root.exists():
        return None

    for revision_dir in sorted(config_root.glob("*"), reverse=True):
        arrow_candidates = sorted(revision_dir.glob("*/*.arrow"))
        if arrow_candidates:
            return str(arrow_candidates[0].resolve())
        parquet_candidates = sorted(revision_dir.glob("*/*.parquet"))
        if len(parquet_candidates) == 1:
            return str(parquet_candidates[0].resolve())
    return None


def localize_benchmark_task(task: fev.Task) -> fev.Task | None:
    if not OFFICIAL_FEV_USE_LOCAL_HF_CACHE or task.dataset_config is None:
        return task

    cached_file = resolve_cached_hf_dataset_file(task.dataset_path, task.dataset_config)
    if cached_file is None:
        if OFFICIAL_FEV_SKIP_UNCACHED_TASKS:
            return None
        return task

    task_config = task.to_dict()
    task_config["dataset_path"] = cached_file
    task_config["dataset_config"] = None
    task_config["task_name"] = task.task_name
    return fev.Task(**task_config)


def resolve_official_fev_benchmark_source() -> str:
    local_roots = [
        Path("fev"),
        Path("external/fev"),
        Path("..") / "fev",
        Path("..") / ".." / "fev",
    ]
    for root in local_roots:
        for relative_path in OFFICIAL_FEV_BENCHMARK_PATH_CANDIDATES:
            candidate = (root / relative_path).resolve()
            if candidate.exists():
                return str(candidate)

    if OFFICIAL_FEV_BENCHMARK_CACHE_PATH.exists():
        return str(OFFICIAL_FEV_BENCHMARK_CACHE_PATH)

    if OFFICIAL_FEV_ALLOW_NETWORK:
        for relative_path in OFFICIAL_FEV_BENCHMARK_PATH_CANDIDATES:
            url = f"https://raw.githubusercontent.com/{OFFICIAL_FEV_REPO}/{OFFICIAL_FEV_BENCHMARK_REF}/{relative_path}"
            try:
                response = http_get_with_retries(url, timeout=20)
                OFFICIAL_FEV_BENCHMARK_CACHE_PATH.write_text(response.text, encoding="utf-8")
                return str(OFFICIAL_FEV_BENCHMARK_CACHE_PATH)
            except requests.RequestException:
                continue

    raise FileNotFoundError(
        "Could not resolve the official fev-bench benchmark definition. "
        "Clone autogluon/fev locally, keep a cached benchmark YAML, or enable GitHub raw access."
    )


def load_official_fev_benchmark(task_limit: int | None = None) -> tuple[fev.Benchmark, str, int, list[str]]:
    benchmark_source = resolve_official_fev_benchmark_source()
    benchmark = fev.Benchmark.from_yaml(benchmark_source)
    if task_limit is not None:
        benchmark = fev.Benchmark(tasks=benchmark.tasks[:task_limit])

    localized_tasks = []
    skipped_tasks = []
    localized_count = 0
    for task in benchmark.tasks:
        localized_task = localize_benchmark_task(task)
        if localized_task is None:
            skipped_tasks.append(task.task_name)
            continue
        if localized_task.dataset_config is None and task.dataset_config is not None:
            localized_count += 1
        localized_tasks.append(localized_task)

    return fev.Benchmark(tasks=localized_tasks), benchmark_source, localized_count, skipped_tasks


def load_official_fev_bench_public_summaries(commit: str = OFFICIAL_FEV_BENCHMARK_REF) -> pd.DataFrame:
    cache_path = OUTPUT_DIR / f"official_fev_bench_results_{commit}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path)

    if not OFFICIAL_FEV_ALLOW_NETWORK:
        print("Public fev-bench summaries unavailable offline; continuing without official comparison rows.")
        return pd.DataFrame()

    api_url = f"https://api.github.com/repos/{OFFICIAL_FEV_REPO}/contents/{OFFICIAL_FEV_RESULTS_PATH}?ref={commit}"
    try:
        response = http_get_with_retries(api_url, timeout=30)
        csv_files = [entry["path"] for entry in response.json() if entry["name"].endswith(".csv")]
    except requests.RequestException:
        print("Could not download official fev-bench public summaries; continuing without official comparison rows.")
        return pd.DataFrame()

    if not csv_files:
        raise FileNotFoundError(f"No public result CSVs found in {OFFICIAL_FEV_RESULTS_PATH} at ref {commit}")

    dataframes = []
    for file_path in tqdm.tqdm(csv_files, desc="Loading official fev-bench summaries"):
        raw_url = f"https://raw.githubusercontent.com/{OFFICIAL_FEV_REPO}/{commit}/{file_path}"
        raw_response = http_get_with_retries(raw_url, timeout=60)
        dataframes.append(pd.read_csv(io.StringIO(raw_response.text)))

    summaries_df = pd.concat(dataframes, ignore_index=True)
    summaries_df.to_csv(cache_path, index=False)
    return summaries_df


def benchmark_pipeline(label: str, pipeline: Chronos2Pipeline, benchmark: fev.Benchmark, batch_size: int = 256):
    summaries = []
    timings = []

    for task in tqdm.tqdm(benchmark.tasks, desc=f"Evaluating {label}"):
        predictions_per_window, inference_time_s = pipeline.predict_fev(task, batch_size=batch_size)
        summary = task.evaluation_summary(predictions_per_window, label)
        summary["model_name"] = label
        summary["model"] = label
        summary["inference_time_s"] = inference_time_s
        summaries.append(summary)
        timings.append({
            "model": label,
            "task_name": task.task_name,
            "dataset": task.dataset_config,
            "inference_time_s": inference_time_s,
        })

    return pd.DataFrame(summaries), pd.DataFrame(timings)


def compute_official_style_leaderboards(
    summaries_df: pd.DataFrame,
    included_models: list[str] | None = None,
    metrics: list[str] | None = None,
) -> dict[str, pd.DataFrame]:
    metrics = metrics or OFFICIAL_FEV_LEADERBOARD_METRICS
    leaderboards = {}
    for metric_name in metrics:
        leaderboard_df = fev.leaderboard(
            summaries=summaries_df.to_dict(orient="records"),
            metric_column=metric_name,
            missing_strategy="impute",
            baseline_model=OFFICIAL_FEV_BASELINE_MODEL,
            leakage_imputation_model=OFFICIAL_FEV_LEAKAGE_IMPUTATION_MODEL,
            normalize_time_per_n_forecasts=OFFICIAL_FEV_NORMALIZE_TIME_PER_N_FORECASTS,
            included_models=included_models,
        ).reset_index()
        leaderboard_df["skill_score"] = leaderboard_df["skill_score"] * 100
        leaderboard_df["win_rate"] = leaderboard_df["win_rate"] * 100
        leaderboards[metric_name] = leaderboard_df.sort_values("win_rate", ascending=False).reset_index(drop=True)
    return leaderboards


baseline_pipeline = build_pipeline_from_graph(baseline_mg, model)
spectral_pipeline = build_pipeline_from_graph(spectral_mg, model)
reference_pipeline = Chronos2Pipeline(model=model)

In [ ]:
official_benchmark, official_benchmark_source, localized_task_count, skipped_task_names = load_official_fev_benchmark(task_limit=FEV_BENCH_TASK_LIMIT)
official_public_summaries = load_official_fev_bench_public_summaries(commit=OFFICIAL_FEV_BENCHMARK_REF)
print(f"Loaded official fev-bench definition from: {official_benchmark_source}")
print(f"Benchmark tasks to evaluate: {len(official_benchmark.tasks)}")
if localized_task_count:
    print(f"Rewrote {localized_task_count} benchmark tasks to local Hugging Face cache files.")
if skipped_task_names:
    raise RuntimeError(
        "Refusing to run a partial fev-bench benchmark. "
        f"Missing local cache for {len(skipped_task_names)} task(s): {skipped_task_names}. "
        "Populate the Hugging Face cache for those tasks before rerunning the benchmark."
    )

fp32_df, fp32_timing = benchmark_pipeline("fp32_chronos2", reference_pipeline, official_benchmark)
baseline_df, baseline_timing = benchmark_pipeline("baseline_ptq_chronos2", baseline_pipeline, official_benchmark)
spectral_df_bench, spectral_timing = benchmark_pipeline("spectral_ptq_chronos2", spectral_pipeline, official_benchmark)

all_summaries = pd.concat([fp32_df, baseline_df, spectral_df_bench], ignore_index=True, sort=False)
all_timings = pd.concat([fp32_timing, baseline_timing, spectral_timing], ignore_index=True)


In [ ]:
canonical_benchmark = fev.Benchmark.from_yaml(resolve_official_fev_benchmark_source())
if FEV_BENCH_TASK_LIMIT is not None:
    canonical_benchmark = fev.Benchmark(tasks=canonical_benchmark.tasks[:FEV_BENCH_TASK_LIMIT])

canonical_task_df = pd.DataFrame(
    [
        {
            "task_name": task.task_name,
            "dataset_path": task.dataset_path,
            "dataset_config": task.dataset_config,
        }
        for task in canonical_benchmark.tasks
    ]
)

normalized_all_summaries = all_summaries.copy()
if "model_name" not in normalized_all_summaries.columns and "model" in normalized_all_summaries.columns:
    normalized_all_summaries = normalized_all_summaries.rename(columns={"model": "model_name"})
normalized_all_summaries = (
    normalized_all_summaries.drop(columns=["dataset_path", "dataset_config"], errors="ignore")
    .merge(canonical_task_df, on="task_name", how="left")
)

normalized_official_public_summaries = official_public_summaries.copy()

task_num_forecasts = normalized_all_summaries.groupby("task_name", as_index=False)["num_forecasts"].first()
normalized_official_public_summaries = (
    normalized_official_public_summaries.drop(columns=["num_forecasts"], errors="ignore")
    .merge(task_num_forecasts, on="task_name", how="left")
)

combined_summaries = pd.concat(
    [normalized_official_public_summaries, normalized_all_summaries],
    ignore_index=True,
    sort=False,
)

focus_models = [
    OFFICIAL_FEV_BASELINE_MODEL,
    OFFICIAL_FEV_LEAKAGE_IMPUTATION_MODEL,
    "fp32_chronos2",
    "baseline_ptq_chronos2",
    "spectral_ptq_chronos2",
]
if official_public_summaries.empty:
    print("Official public fev-bench summaries are unavailable, so leaderboard comparisons are skipped.")
    official_style_leaderboards = {}
    focused_official_style_leaderboards = {}
else:
    official_style_leaderboards = compute_official_style_leaderboards(combined_summaries)
    focused_official_style_leaderboards = compute_official_style_leaderboards(
        combined_summaries,
        included_models=focus_models,
    )

display(all_summaries)
display(all_timings)

for metric_name in OFFICIAL_FEV_LEADERBOARD_METRICS:
    if metric_name not in official_style_leaderboards:
        continue
    print(f"Official-style fev-bench leaderboard ({metric_name})")
    display(official_style_leaderboards[metric_name])
    print(f"Focused comparison ({metric_name})")
    display(focused_official_style_leaderboards[metric_name])


In [ ]:
all_timings.groupby("model", as_index=False)["inference_time_s"].mean()

In [ ]:
from pathlib import Path

results_dir = OUTPUT_DIR / "tables"
results_dir.mkdir(parents=True, exist_ok=True)

all_summaries.to_csv(results_dir / "all_summaries.csv", index=False)

all_timings.to_csv(results_dir / "all_timings.csv", index=False)

for metric_name, df in official_style_leaderboards.items():
    df.to_csv(results_dir / f"leaderboard_{metric_name}.csv", index=False)

for metric_name, df in focused_official_style_leaderboards.items():
    df.to_csv(results_dir / f"leaderboard_focused_{metric_name}.csv", index=False)


![Spectral Results](artifacts/spectral_imgs/SpectralQuantResults.jpg)